# Section 2: Build Snowball2 training files and train operation models

This notebook prepares Snowball2 training artifacts from the annotated Excel
training sheet written by notebook 000.

## Input

- Excel file stem: `training_events_all_operations`
- Excel sheet: `Training_events`
- 13 columns: `ID`, `year`, the operation annotation, and the temperature/time
  annotations. Document context (`title`, `paragraph_text`) and the unused
  `synthesis_method` / `material_text_span` columns are not part of this pipeline;
  a paper is identified by `ID` and `year`.

## Main outputs

- `snowball2_annotations_all_years.json`
- `positive_operation_events.csv`
- `snowball2_pickled_list.pkl`
- `operation_model.py`
- `OperationTemperature_list.pkl`
- `Snowball_model/OperationTemperature.pkl`
- `OperationTemperature_seed_rejects.json`
- `OperationTime_list.pkl`
- `Snowball_model/OperationTime.pkl`
- `OperationTime_seed_rejects.json`

## Framework logic

1. Load the manually annotated training rows.
2. Keep every row that has an operation event and at least one COMPLETE
   temperature or time annotation (value **and** unit). The sheet is already
   complete, so in practice no rows are dropped here.
3. Store each event with its operation type, annotated operation trigger,
   event span, temperature annotation, time annotation, and `ID` / `year`.
4. Save one inspection CSV and one Snowball2 pickle.
5. Generate `operation_model.py` from the annotated operation triggers.
6. Train `OperationTemperature` from the rows with a complete temperature.
7. Train `OperationTime` from the rows with a complete time.
8. Save reject logs and check the expected output files.

The goal is that the parser connects a temperature and a time to the correct
operation specifier, rather than extracting free-floating values.


In [ ]:
# Shared notebook controls: keep normal runs compact, but allow detailed previews when needed.
import pandas as pd

RUN_VERBOSE = False


def display_table(rows):
    """Show compact status tables in notebooks; fall back to text output in scripts."""
    table = pd.DataFrame(rows)

    try:
        display(table)
    except NameError:
        print(table.to_string(index=False))

    return table

## Part 1:

# Cell 1 — Build one combined Snowball2 JSON from the combined Excel sheet

In [ ]:
# Cell 1: Load the annotated Excel sheet and convert training rows into Snowball2 JSON.

import json
from pathlib import Path

import pandas as pd


# ---------------------------------------------------------------------
# Framework contract: keep these names unchanged.
# Other notebooks/scripts expect this Excel input, sheet name, and JSON output.
# ---------------------------------------------------------------------

excel_stem = "training_events_all_operations"
sheet_name = "Training_events"

combined_json_path = Path("snowball2_annotations_all_years.json")


# Finds the file whether you typed the stem only or saved it as .xlsx/.xlsm/.xls
def find_excel_file(file_stem_or_path):
    candidates = [
        Path(f"{file_stem_or_path}.xlsx"),
        Path(f"{file_stem_or_path}.xlsm"),
        Path(f"{file_stem_or_path}.xls"),
        Path(file_stem_or_path),
    ]

    path = next((p for p in candidates if p.is_file()), None)

    if path is None:
        raise FileNotFoundError(
            "Could not find the Excel file. Checked: "
            + ", ".join(str(p) for p in candidates)
        )

    return path


excel_path = find_excel_file(excel_stem)

read_excel_kwargs = {
    "sheet_name": sheet_name,
    "dtype": object,
}

# Useful only if the file exists without an Excel extension
if excel_path.suffix == "":
    read_excel_kwargs["engine"] = "openpyxl"

df = pd.read_excel(excel_path, **read_excel_kwargs)
df.columns = [str(c).strip() for c in df.columns]

# Training reads event_text_span only; document context (title, paragraph_text)
# is deliberately not carried into this pipeline. ID and year are required
# instead, and are the only link from an event back to its source paper.
required_cols = [
    "ID",
    "year",
    "event_text_span",
    "operation_type",
    "operation_verb_text_span",
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df.dropna(how="all").copy()
df["_source_excel_row"] = df.index + 2


def clean_scalar(value):
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass

    if isinstance(value, str):
        value = value.strip()
        return None if value == "" else value

    if hasattr(value, "item"):
        try:
            value = value.item()
        except Exception:
            pass

    if isinstance(value, float) and value.is_integer():
        return int(value)

    return value


def get_cell(row, col):
    if col is None or col not in row.index:
        return None

    return clean_scalar(row[col])


def compact_obj(obj):
    return {
        k: v
        for k, v in obj.items()
        if v is not None and v != {}
    }


def make_text_obj(row, col):
    return compact_obj({
        "text": get_cell(row, col)
    })


def make_parameter(row, text_col, value_col, unit_col, qualifier_col):
    return compact_obj({
        "text": get_cell(row, text_col),
        "value": get_cell(row, value_col),
        "unit": get_cell(row, unit_col),
        "qualifier": get_cell(row, qualifier_col),
    })


def has_complete_value_and_unit(row, value_col, unit_col):
    """A parameter counts as annotated only when BOTH value and unit are present.

    Text spans and qualifiers are never turned into entities, so a row carrying
    only a qualifier ("room temperature", "overnight") or only a text span
    cannot produce a Snowball phrase for that parameter. Temperature and time
    are judged independently: a row with a complete time and no temperature is
    still a valid training row for OperationTime.
    """
    return all(
        get_cell(row, col) is not None
        for col in [value_col, unit_col]
    )


def make_event(row, event_id):
    return compact_obj({
        "_source_excel_row": get_cell(row, "_source_excel_row"),
        
        "event_id": event_id,
        # Fixed Snowball2 shape requirement, not a label that ever varies.
        "event_role": "target_operation",

        # This is taken directly from the annotated training data.
        # No top-N filtering, no priority, and no external operation vocabulary.
        "operation_type": get_cell(row, "operation_type"),
        "specifier": make_text_obj(row, "operation_verb_text_span"),

        # No material entity: the framework targets the operation/temperature/time
        # relation only, and the material column is not part of the training sheet.
        "event_text_span": get_cell(row, "event_text_span"),

        # Rule R-T2: the vessel the step takes place in. It is not part of any
        # Snowball phrase; it is carried to positive_operation_events.csv, from
        # which notebook 002 builds its vessel vocabulary.
        "vessel": make_text_obj(row, "vessel_text_span"),

        "temperature": make_parameter(
            row,
            text_col="temperature_text_span",
            value_col="temperature_value",
            unit_col="temperature_unit",
            qualifier_col="temperature_qualifier",
        ),

        "time": make_parameter(
            row,
            text_col="time_text_span",
            value_col="time_value",
            unit_col="time_unit",
            qualifier_col="time_qualifier",
        ),
    })


# Source of truth: all operation events are already annotated in the training Excel sheet.
# Rows are kept when they have:
#   1. event_text_span
#   2. operation_type
#   3. operation_verb_text_span
#   4. at least one COMPLETE temperature or time annotation (value AND unit)
records = []

skipped_counts = {
    "missing_event_text_span": 0,
    "missing_operation_type": 0,
    "missing_operation_verb_text_span": 0,
    "without_temperature_or_time_annotation": 0,
}

for _, row in df.sort_values("_source_excel_row").iterrows():
    sentence = get_cell(row, "event_text_span")
    operation_type = get_cell(row, "operation_type")
    operation_verb = get_cell(row, "operation_verb_text_span")

    if sentence is None:
        skipped_counts["missing_event_text_span"] += 1
        continue

    if operation_type is None:
        skipped_counts["missing_operation_type"] += 1
        continue

    if operation_verb is None:
        skipped_counts["missing_operation_verb_text_span"] += 1
        continue

    has_temperature = has_complete_value_and_unit(
        row,
        value_col="temperature_value",
        unit_col="temperature_unit",
    )

    has_time = has_complete_value_and_unit(
        row,
        value_col="time_value",
        unit_col="time_unit",
    )

    if not (has_temperature or has_time):
        skipped_counts["without_temperature_or_time_annotation"] += 1
        continue

    records.append(row)

paragraph_files = []

for record_number, row in enumerate(records, start=1):
    record_id = f"P{record_number}"
    event = make_event(row, record_id)

    content = compact_obj({
        "paragraph_id": record_id,
        # Fixed Snowball2 shape requirement: there is no negative class
        # anywhere in this pipeline, every record is a positive example.
        "paragraph_label": "positive",
        # Paper-level traceability, kept in place of title/paragraph_text.
        "ID": get_cell(row, "ID"),
        "year": get_cell(row, "year"),
        "operation_events": [event],
    })

    paragraph_files.append({
        "filename": f"{record_id}.snowball2_annotation.json",
        "content": content,
    })


combined_data = {
    "paragraph_files": paragraph_files
}

with combined_json_path.open("w", encoding="utf-8") as f:
    json.dump(combined_data, f, ensure_ascii=False, indent=2, default=str)


included_operation_types = sorted({
    str(get_cell(row, "operation_type"))
    for row in records
    if get_cell(row, "operation_type") is not None
})

included_operation_triggers = sorted({
    str(get_cell(row, "operation_verb_text_span"))
    for row in records
    if get_cell(row, "operation_verb_text_span") is not None
})

total_events = sum(
    len(pf["content"].get("operation_events", []))
    for pf in paragraph_files
)


display_table([
    {"item": "input_excel_file", "value": str(excel_path)},
    {"item": "output_json", "value": str(combined_json_path)},
    {"item": "operation_event_policy", "value": "all valid annotated operation events from training data"},
    {"item": "total_training_records_included", "value": len(paragraph_files)},
    {"item": "total_operation_events", "value": total_events},
    {"item": "unique_operation_types_included", "value": len(included_operation_types)},
    {"item": "unique_operation_triggers_included", "value": len(included_operation_triggers)},
    # The annotation sheet is already complete, so these four counters are
    # expected to read 0: no rows are dropped at this stage. They are kept as
    # evidence of that completeness, not as a filter that does work.
    {"item": "skipped_missing_event_text_span", "value": skipped_counts["missing_event_text_span"]},
    {"item": "skipped_missing_operation_type", "value": skipped_counts["missing_operation_type"]},
    {"item": "skipped_missing_operation_verb_text_span", "value": skipped_counts["missing_operation_verb_text_span"]},
    {"item": "skipped_without_temperature_or_time_annotation", "value": skipped_counts["without_temperature_or_time_annotation"]},
])

if globals().get("RUN_VERBOSE", False):
    operation_type_counts = (
        pd.Series(
            [
                get_cell(row, "operation_type")
                for row in records
                if get_cell(row, "operation_type") is not None
            ],
            name="operation_type",
        )
        .astype(str)
        .value_counts()
        .rename_axis("operation_type")
        .reset_index(name="count")
    )

    display(operation_type_counts)

### Cell 1 output check

Expected result:

- Input Excel file: `training_events_all_operations.xlsx`, sheet `Training_events`
- Output JSON: `snowball2_annotations_all_years.json`
- One record per annotated operation event.
- All four `skipped_*` counters read 0: the annotation sheet is already complete,
  so this stage drops no rows.


# Cell 2 — Read and flatten the JSON

In [ ]:
# Cell 2: Flatten Snowball2 JSON back into one event-level table for review and CSV export.

import json
from pathlib import Path

import pandas as pd


json_path = Path("snowball2_annotations_all_years.json")

with json_path.open("r", encoding="utf-8") as f:
    data = json.load(f)


def nested_get(obj, *keys):
    current = obj

    for key in keys:
        if not isinstance(current, dict):
            return None

        current = current.get(key)

    return current


def get_events(content):
    if isinstance(content.get("operation_events"), list):
        return content["operation_events"]
    return []


flattened_columns = [
    "filename",
    "paragraph_id",
    "paragraph_label",

    "ID",
    "year",

    "event_id",
    "event_role",
    "source_excel_row",

    "operation_type",
    "operation_verb_text_span",

    "event_text_span",
    "vessel_text_span",

    "temperature_text_span",
    "temperature_value",
    "temperature_unit",
    "temperature_qualifier",

    "time_text_span",
    "time_value",
    "time_unit",
    "time_qualifier",
]

positive_rows = []

for pf in data["paragraph_files"]:
    filename = pf["filename"]
    content = pf["content"]

    paragraph_id = content.get("paragraph_id")
    paragraph_label = content.get("paragraph_label")
    paper_id = content.get("ID")
    paper_year = content.get("year")

    for ev in get_events(content):
        row = {
            "filename": filename,
            "paragraph_id": paragraph_id,
            "paragraph_label": paragraph_label,
            "ID": paper_id,
            "year": paper_year,

            "event_id": ev.get("event_id"),
            "event_role": ev.get("event_role"),
            "source_excel_row": ev.get("_source_excel_row"),

            "operation_type": ev.get("operation_type"),
            "operation_verb_text_span": nested_get(ev, "specifier", "text"),

            "event_text_span": ev.get("event_text_span"),
            "vessel_text_span": nested_get(ev, "vessel", "text"),

            "temperature_text_span": nested_get(ev, "temperature", "text"),
            "temperature_value": nested_get(ev, "temperature", "value"),
            "temperature_unit": nested_get(ev, "temperature", "unit"),
            "temperature_qualifier": nested_get(ev, "temperature", "qualifier"),

            "time_text_span": nested_get(ev, "time", "text"),
            "time_value": nested_get(ev, "time", "value"),
            "time_unit": nested_get(ev, "time", "unit"),
            "time_qualifier": nested_get(ev, "time", "qualifier"),
        }

        positive_rows.append(row)


positive_df = pd.DataFrame(positive_rows, columns=flattened_columns)

### Cell 2 Printing: for easier visibility 
# Compact inspection summary for the flattened event table.

display_table([
    {"metric": "positive_operation_events", "value": len(positive_df)},
    {
        "metric": "unique_paragraphs",
        "value": positive_df["paragraph_id"].nunique() if "paragraph_id" in positive_df else None,
    },
    {
        "metric": "unique_operation_types",
        "value": positive_df["operation_type"].nunique(dropna=True) if "operation_type" in positive_df else None,
    },
    {
        "metric": "unique_operation_triggers",
        "value": positive_df["operation_verb_text_span"].nunique(dropna=True) if "operation_verb_text_span" in positive_df else None,
    },
])

if globals().get("RUN_VERBOSE", False):
    display_columns = [
        "paragraph_id",
        "ID",
        "year",
        "source_excel_row",
        "event_id",
        "operation_type",
        "operation_verb_text_span",
        "event_text_span",
        "temperature_text_span",
        "temperature_value",
        "temperature_unit",
        "time_text_span",
        "time_value",
        "time_unit",
    ]

    available_display_columns = [
        col for col in display_columns
        if col in positive_df.columns
    ]

    display(positive_df[available_display_columns].head(10))

    display(
        positive_df["operation_type"]
        .value_counts(dropna=False)
        .rename_axis("operation_type")
        .reset_index(name="count")
    )

# Cell 2A — Basic annotation validation checks

Purpose: catch obvious data issues before saving the pickle or training.

In [ ]:
# Cell 2A: Validation checks on the flattened event table.
#
# Structural integrity only. Annotation completeness is reported separately in
# Cell 2B, so the two cells no longer compute the same thing twice.

if positive_df.empty:
    raise ValueError(
        "No positive operation events were created. "
        "Check event_text_span, operation_type, operation_verb_text_span, temperature, and time columns."
    )


def missing_count(df, column):
    return int(df[column].isna().sum()) if column in df.columns else None


duplicate_check_columns = [
    col for col in [
        "ID",
        "operation_type",
        "operation_verb_text_span",
        "event_text_span",
        "temperature_value",
        "temperature_unit",
        "time_value",
        "time_unit",
    ]
    if col in positive_df.columns
]

# Repeated identical annotations within one paper are expected: a paper often
# describes the same operation under the same conditions for several samples.
# They are counted here for transparency, not removed.
duplicate_count = (
    positive_df.duplicated(subset=duplicate_check_columns).sum()
    if duplicate_check_columns
    else 0
)

id_mismatch_count = (
    (positive_df["paragraph_id"] != positive_df["event_id"]).sum()
    if {"paragraph_id", "event_id"}.issubset(positive_df.columns)
    else None
)

display_table([
    {"check": "total_operation_events", "value": len(positive_df)},
    {"check": "unique_papers (ID)", "value": positive_df["ID"].nunique(dropna=True)},
    {"check": "unique_years", "value": positive_df["year"].nunique(dropna=True)},
    {"check": "unique_operation_types", "value": positive_df["operation_type"].nunique(dropna=True)},
    {"check": "unique_operation_triggers", "value": positive_df["operation_verb_text_span"].nunique(dropna=True)},
    {"check": "repeated_identical_annotations", "value": int(duplicate_count)},
    {"check": "paragraph_id_event_id_mismatches", "value": int(id_mismatch_count) if id_mismatch_count is not None else None},
    {"check": "missing_ID", "value": missing_count(positive_df, "ID")},
    {"check": "missing_year", "value": missing_count(positive_df, "year")},
    {"check": "missing_operation_type", "value": missing_count(positive_df, "operation_type")},
    {"check": "missing_operation_verb_text_span", "value": missing_count(positive_df, "operation_verb_text_span")},
    {"check": "missing_event_text_span", "value": missing_count(positive_df, "event_text_span")},
    {"check": "missing_source_excel_row", "value": missing_count(positive_df, "source_excel_row")},
])


In [ ]:
# Cell 2B: Annotation completeness, per parameter.
#
# Each parameter (temperature, time) falls into exactly one of three states:
#
#   numeric          value AND unit are annotated -> can become a training phrase
#   non-numeric      the parameter is mentioned in the text and annotated as a
#                    text span + qualifier, but the source gives no number
#                    ("room temperature", "overnight", "until a gel was formed")
#   not mentioned    the operation reports no temperature / no time at all
#
# Only "numeric" can be trained: a Snowball phrase needs a value token and a unit
# token to point at. The non-numeric rows are correct annotations of vague source
# text, not missing work, and they still train the OTHER parameter's model.

def has_nonempty_value(df, column):
    if column not in df.columns:
        return pd.Series(False, index=df.index)

    return (
        df[column]
        .notna()
        & df[column].astype(str).str.strip().ne("")
    )


def parameter_states(df, value_col, unit_col, text_col):
    numeric = has_nonempty_value(df, value_col) & has_nonempty_value(df, unit_col)
    mentioned = has_nonempty_value(df, text_col) | numeric

    return numeric, (mentioned & ~numeric), ~mentioned


temperature_numeric, temperature_non_numeric, temperature_absent = parameter_states(
    positive_df, "temperature_value", "temperature_unit", "temperature_text_span"
)

time_numeric, time_non_numeric, time_absent = parameter_states(
    positive_df, "time_value", "time_unit", "time_text_span"
)

total = len(positive_df)

display_table([
    {
        "parameter": "temperature",
        "numeric (trainable)": int(temperature_numeric.sum()),
        "non-numeric mention": int(temperature_non_numeric.sum()),
        "not mentioned": int(temperature_absent.sum()),
        "total events": total,
    },
    {
        "parameter": "time",
        "numeric (trainable)": int(time_numeric.sum()),
        "non-numeric mention": int(time_non_numeric.sum()),
        "not mentioned": int(time_absent.sum()),
        "total events": total,
    },
])

# How the same 669 events split across the two models.
display_table([
    {"coverage": "events_total", "value": total},
    {"coverage": "trains_both_models", "value": int((temperature_numeric & time_numeric).sum())},
    {"coverage": "trains_temperature_only", "value": int((temperature_numeric & ~time_numeric).sum())},
    {"coverage": "trains_time_only", "value": int((time_numeric & ~temperature_numeric).sum())},
    {"coverage": "trains_neither", "value": int((~(temperature_numeric | time_numeric)).sum())},
])


# Cell 3 — Save flattened tables and Snowball2 pickled list

In [ ]:
# Cell 3: Save the positive-event CSV and paragraph-level Snowball2 pickle.

import json
import pickle
from pathlib import Path


positive_df.to_csv("positive_operation_events.csv", index=False, encoding="utf-8")

json_path = Path("snowball2_annotations_all_years.json")

# Always reload the JSON so the pickle matches the current Cell 1 output.
with json_path.open("r", encoding="utf-8") as f:
    data = json.load(f)


def get_events_for_pickle(content):
    if isinstance(content.get("operation_events"), list):
        return content["operation_events"]

    return []

# Keep paragraph-style wrapper because later training cells expect
# each record to contain an "events" list.
# Important:
# No operation_type filtering is applied here.
# The pickle keeps all operation events already written by Cell 1.
snowball2_list = []

for pf in data["paragraph_files"]:
    content = pf["content"]
    events = get_events_for_pickle(content)

    snowball2_list.append({
        "filename": pf.get("filename"),
        "paragraph_id": content.get("paragraph_id"),
        "ID": content.get("ID"),
        "year": content.get("year"),
        "paragraph_label": content.get("paragraph_label"),
        "events": events,
    })


pickle_path = Path("snowball2_pickled_list.pkl")

with pickle_path.open("wb") as f:
    pickle.dump(snowball2_list, f, protocol=pickle.HIGHEST_PROTOCOL)


total_pickle_events = sum(
    len(item.get("events", []))
    for item in snowball2_list
)

pickle_operation_types = sorted({
    ev.get("operation_type")
    for item in snowball2_list
    for ev in item.get("events", [])
    if ev.get("operation_type") is not None
})

records_with_paper_id = sum(
    1
    for item in snowball2_list
    if item.get("ID") is not None
)

display_table([
    {"output": "positive_operation_events.csv", "status": "saved"},
    {"output": str(pickle_path), "status": "saved"},
    {"output": "snowball2_list_length", "status": len(snowball2_list)},
    {"output": "records_with_paper_id", "status": records_with_paper_id},
    {"output": "total_operation_events_in_pickle", "status": total_pickle_events},
    {"output": "unique_operation_types_in_pickle", "status": len(pickle_operation_types)},
])

# Part 2:

## Cell 4 — Write ChemDataExtractor / Snowball2 schema

This cell writes `operation_model.py`.

In [ ]:
# Cell 4: Write the ChemDataExtractor/Snowball2 model schema used by training.
#
# The operation vocabulary is derived entirely from the annotations: every term
# below is an annotated operation_verb_text_span (or a spacing/hyphenation
# variant of one). No external verb lexicon and no hand-written rule is used.
#
# Reads positive_operation_events.csv rather than the in-memory positive_df, so
# this cell can be run on its own after Part 1 has been run once.

from pathlib import Path
from pprint import pformat

import pandas as pd


POSITIVE_CSV_PATH = Path("positive_operation_events.csv")

if not POSITIVE_CSV_PATH.exists():
    raise FileNotFoundError(
        f"{POSITIVE_CSV_PATH} not found. Run Part 1 (Cells 1-3) first."
    )

operation_events = pd.read_csv(POSITIVE_CSV_PATH, dtype=str)

if "operation_verb_text_span" not in operation_events.columns:
    raise ValueError(
        f"{POSITIVE_CSV_PATH} is missing the operation_verb_text_span column."
    )


def operation_surface_variants(term):
    """Return surface variants of an annotated operation trigger.

    This does not introduce new operation types. It only handles
    spacing/hyphenation variants of operation_verb_text_span.
    """
    term = str(term).strip()

    if not term:
        return []

    variants = [term]

    if "-" in term:
        variants.append(term.replace("-", " "))

    if " " in term:
        variants.append(term.replace(" ", "-"))

    return variants


def order_operation_terms(terms):
    """Order the alternation so the longest trigger wins.

    ChemDataExtractor builds `a | b` as First([a, b]), which takes the FIRST
    alternative that matches, not the longest. In alphabetical order a short
    trigger therefore hides every longer trigger starting with the same word
    ("calcination" would hide "calcination was", "stirred" would hide
    "stirred continuously"). Ordering by token count, then by length, makes the
    most specific trigger match first.

    I() is case-insensitive, so terms differing only in case are the same
    matcher and only the first is kept.
    """
    unique_terms = []
    seen = set()

    for term in terms:
        term = str(term).strip()
        key = term.lower()

        if not term or key in seen:
            continue

        unique_terms.append(term)
        seen.add(key)

    return sorted(
        unique_terms,
        key=lambda t: (-len(t.split()), -len(t), t.lower()),
    )


operation_terms = order_operation_terms(
    variant
    for value in operation_events["operation_verb_text_span"].dropna()
    for variant in operation_surface_variants(value)
)

if not operation_terms:
    raise ValueError("No operation terms found in operation_verb_text_span.")

operation_model_code = f'''\
import re

from chemdataextractor.model import StringType, ModelType, Compound

try:
    from chemdataextractor.model.units.temperature import TemperatureModel
except ImportError:
    from chemdataextractor.model import TemperatureModel

try:
    from chemdataextractor.model.units.time import TimeModel
except ImportError:
    from chemdataextractor.model import TimeModel

try:
    from chemdataextractor.parse import I
    from chemdataextractor.parse.actions import join
except ImportError:
    from chemdataextractor.parse.elements import I
    from chemdataextractor.parse.actions import join


# Generated from the annotated operation triggers. ORDER IS SIGNIFICANT: the
# alternation is a First(), so the list runs longest-trigger-first and the most
# specific match wins.
_OPERATION_SPECIFIER_TERMS = {pformat(operation_terms, width=88)}

def phrase_expression(term):
    tokens = [token for token in re.split(r"\s+", term.strip()) if token]

    if not tokens:
        raise ValueError("Empty operation term.")

    expression = I(tokens[0])

    for token in tokens[1:]:
        expression = expression + I(token)

    return expression


def operation_expression(terms):
    expressions = [phrase_expression(term) for term in terms]

    if not expressions:
        raise ValueError("No operation terms provided.")

    expression = expressions[0]

    for next_expression in expressions[1:]:
        expression = expression | next_expression

    return expression


temperature_specifier_expression = operation_expression(_OPERATION_SPECIFIER_TERMS).add_action(join)
time_specifier_expression = operation_expression(_OPERATION_SPECIFIER_TERMS).add_action(join)


class OperationTemperature(TemperatureModel):
    """
    Snowball2 model for operation-related temperatures.

    Target entities:
        specifier -> annotated operation_verb_text_span
        raw_value -> temperature_value
        raw_units -> temperature_unit
    """

    specifier = StringType(
        parse_expression=temperature_specifier_expression,
        required=True,
        contextual=False,
        updatable=True,
    )

    # Never populated by this framework: no compound/material entity is built.
    # Kept because Snowball2 expands the model's nested fields and its candidate
    # generation refers to compound__* tags; removing it breaks the library.
    compound = ModelType(
        Compound,
        required=False,
        contextual=True,
    )


class OperationTime(TimeModel):
    """
    Snowball2 model for operation-related durations/times.

    Target entities:
        specifier -> annotated operation_verb_text_span
        raw_value -> time_value
        raw_units -> time_unit
    """

    specifier = StringType(
        parse_expression=time_specifier_expression,
        required=True,
        contextual=False,
        updatable=True,
    )

    # See the note on OperationTemperature.compound.
    compound = ModelType(
        Compound,
        required=False,
        contextual=True,
    )
'''

Path("operation_model.py").write_text(operation_model_code, encoding="utf-8")

annotated_terms = (
    operation_events["operation_verb_text_span"].dropna().str.strip().nunique()
)

display_table([
    {"item": "operation_model.py", "status": "saved"},
    {"item": "annotated_triggers", "status": int(annotated_terms)},
    {"item": "operation_terms_written", "status": len(operation_terms)},
    {"item": "multi_word_terms", "status": sum(1 for t in operation_terms if len(t.split()) > 1)},
])


In [ ]:
# Cell 4 check: Confirm the Part 1 / Part 2 artifacts exist.
# (The Part 3 training outputs are checked separately at the end of the notebook.)

from pathlib import Path

check_paths = [
    Path("operation_model.py"),
    Path("snowball2_pickled_list.pkl"),
    Path("positive_operation_events.csv"),
]

display_table([
    {"file": str(path), "status": "FOUND" if path.exists() else "MISSING"}
    for path in check_paths
])

missing = [path for path in check_paths if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Missing expected files: " + ", ".join(str(path) for path in missing)
    )


# Part 3

### Train / cluster Snowball2 from the paragraph-level pickle created in the previous step.

## Cell 1 — Script purpose and file-level documentation

Part 3 turns the paragraph-level pickle written by Part 1 into Snowball2
`Phrase` objects, clusters them, and saves the trained Snowball2 outputs.

Temperature and time are trained by the **same code**: everything that differs
between the two parameters lives in a `ParamSpec` (Cell 13), and both runs call
the same `build_phrase_from_record` and `train`. Only the spec changes.

A training phrase is created only when the whole triple can be located in one
token sequence: the annotated operation specifier, the parameter value, and the
parameter unit. A record that fails any of the three is written to the reject
log with a **reject class**, so the log separates the two very different things
it holds:

- `expected_no_annotation` — this event has no temperature (or no time)
  annotated. Expected under the framework: the two parameters are extracted
  independently, so a time-only event is a result, not a loss.
- `locate_failure` — the parameter IS annotated but could not be found in the
  text. This is the number that measures the seed builder.
- `annotation_gap` — the operation trigger itself is missing. Should be zero.

Material is not an entity, qualifiers are not entities, and no specifier is
inferred from unannotated context: the target relation is
operation / temperature / time only.

Input:

- `snowball2_pickled_list.pkl` (Part 1, Cell 3)

Requires:

- `operation_model.py` (Part 2, Cell 4)

Outputs — temperature (Cell 17):

- `OperationTemperature_list.pkl`
- `Snowball_model/OperationTemperature.pkl`
- `OperationTemperature_seed_rejects.json`

Outputs — time (Cell 18):

- `OperationTime_list.pkl`
- `Snowball_model/OperationTime.pkl`
- `OperationTime_seed_rejects.json`

In [ ]:
# Cell 2: Imports for the training code, with ChemDataExtractor version fallbacks.
#
# Part 3 depends on two artifacts already being on disk:
#   - snowball2_pickled_list.pkl  (Part 1, Cell 3)
#   - operation_model.py          (Part 2, Cell 4)
# The model classes are imported from the generated file, so re-running Part 2
# and then this cell is what picks up a changed operation vocabulary.

import copy
import json
import pickle
import re
from pathlib import Path
from collections import Counter
from dataclasses import dataclass
from typing import Callable, Tuple

# Seed records are shuffled once with a fixed seed before training (Cell 3).
import random

# Sentence moved between chemdataextractor.doc and chemdataextractor.doc.text
# across versions; either import gives the same tokenizer.
try:
    from chemdataextractor.doc import Sentence
except ImportError:
    from chemdataextractor.doc.text import Sentence

from chemdataextractor.model import Compound
# relex is the Snowball implementation shipped inside ChemDataExtractor:
#   Snowball  - training / clustering driver
#   Entity    - one located span (text + tag + token start/end)
#   Relation  - the entity triple a phrase is built around
#   Phrase    - one training example (tokens + relation + context window)
from chemdataextractor.relex.snowball import Snowball
from chemdataextractor.relex.entity import Entity
from chemdataextractor.relex.relationship import Relation
from chemdataextractor.relex.phrase import Phrase

# Written by Part 2 from the annotated operation triggers.
from operation_model import OperationTemperature, OperationTime

In [ ]:
# Cell 3: Runtime configuration shared by both training runs.
#
# Framework contract: one settings block, two parameter specs (Cell 13). Every
# output filename is derived from the spec's name, and notebook 003 loads the
# trained models back from exactly those paths - keep the names unchanged.

INPUT_PATH = Path("snowball2_pickled_list.pkl")
SAVE_DIR = Path("Snowball_model")

# Clustering controls: TSIM_H is the similarity above which a phrase joins an
# existing cluster, CLUSTER_METHOD is how a phrase is scored against a cluster
# that already holds several members.
TSIM_H = 0.95
CLUSTER_METHOD = "max"

# Tokens of context kept on each side of the relation. This window is what a
# learned pattern is built from, so it also limits how much surrounding text a
# pattern can rely on at extraction time.
PREFIX_LENGTH = 1
SUFFIX_LENGTH = 1

# These four values are FIXED, not swept: they are the settings the reported
# models were trained with. Changing any of them invalidates the models that
# notebook 003 evaluates, so change them only together with a re-run of 002/003.

# How far after the value the unit may sit before the search widens to the whole
# sentence. Five tokens covers "500 °C" and "500 ( °C )".
UNIT_SEARCH_WINDOW = 5

# Sentence scoring, used only for records that carry no annotated event span.
# An annotated candidate is worth more than a regex hit, so the regex can break
# a tie but never outvote the annotation.
SENTENCE_SCORE_ANNOTATED = 3
SENTENCE_SCORE_REGEX = 1

# Tokenization repair. The vendored ChemDataExtractor tokenizer leaves a number
# glued to its unit ("4h", "30min") as ONE token - its QUANTITY_RE "split units
# off the end of a numeric value" block is commented out in
# chemdataextractor/nlp/tokenize.py - so an annotated value of "4" can never be
# located in such a sentence and the row is rejected. Splitting them apart
# before tokenizing recovers those seeds.
#
# NOTE: notebook 002 must apply the same normalisation before parsing. The
# patterns learned here are sequences of tokens; if inference tokenizes "4h" as
# one token, no pattern trained on "4" + "h" can fire on it.
SPLIT_GLUED_VALUE_UNITS = True

# Reject payload size: a reject entry keeps the full annotated event for the
# first few records of each reason and identifiers only after that. Without the
# cap the two reject logs are ~1 MB of repeated event dicts.
MAX_FULL_EVENTS_PER_REASON = 5

# Clustering is order-dependent: the first phrase of a cluster seeds it. The
# pickle arrives grouped by paper and year, so the list is shuffled once to keep
# that ordering out of the result; the fixed seed keeps the run reproducible,
# and both models see the records in the same order.
SHUFFLE_SEED_RECORDS = True
SEED_RECORDS_RANDOM_SEED = 42

In [ ]:
# Cell 4: Regex patterns for temperature and time values/units, and the guards.
#
# These regexes only READ text that an annotation already points at (the
# annotated span, or the sentence the record resolves to). They never decide on
# their own that a sentence is a training example. Operation triggers are not
# matched by regex at all: they come from the annotated vocabulary written into
# operation_model.py.

# One number, or a range written with a hyphen/dash or "to"
# ("500", "-20.5", "400-500", "400 to 500"). A range is captured as a single
# value string, the way the annotation records it.
VALUE_PATTERN = (
    r"[-+]?\d+(?:\.\d+)?"
    r"(?:\s*(?:-|–|—|to)\s*[-+]?\d+(?:\.\d+)?)?"
)

# Every Celsius/Kelvin spelling that appears in the source papers. This is a
# reader for source text, not a normalizer: clean_temperature_units (Cell 9)
# decides what the entity finally carries.
TEMP_UNIT_PATTERN = (
    r"(?:"
    r"[°º]\s*[Cc]|"                 # °C, ° C, ºC, º C
    r"℃|"                           # single-character Celsius symbol
    r"[oO]\s*[Cc]|"                 # oC, o C, OC
    r"(?:[Dd]egrees?|[Dd]eg)\s*[Cc]|"
    r"[Cc]elsius|"
    r"[Cc]|"                        # bare C
    r"K|[Kk]elvin"                  # bare K only uppercase; kelvin word can vary
    r")"
)

# A bare "C" or "K" has to end the token, otherwise "5 Ca" or "300 Kg" would be
# read as a temperature. The set deliberately excludes "/", which is what keeps
# RAMP RATES out: "5 °C/min" does not match, while "800 °C" in the same sentence
# does. That exclusion only covers text this regex reads (annotated spans and
# fallback sentence scoring) - it is not the ramp guard for the located span,
# which is is_ramp_unit() in Cell 7, nor for inference, where notebook 002 uses
# the model's own unit parser rather than this pattern.
TEMP_UNIT_BOUNDARY = r"(?=$|[\s,.;:)\]])"

TEMP_RE = re.compile(
    rf"(?P<value>{VALUE_PATTERN})"
    rf"\s*"
    rf"(?P<unit>{TEMP_UNIT_PATTERN})"
    rf"{TEMP_UNIT_BOUNDARY}"
)

# Same value shape for time. The unit alternation ends at a word boundary, which
# is what keeps the one-letter abbreviations honest: "5 min" matches, "5 mm"
# does not, and "3 d" is a duration while "3 dm" is not.
TIME_UNIT_PATTERN = (
    r"h|hr|hrs|hour|hours|"
    r"min|mins|minute|minutes|"
    r"s|sec|secs|second|seconds|"
    r"d|day|days"
)

TIME_RE = re.compile(
    rf"(?P<value>{VALUE_PATTERN})"
    rf"\s*"
    rf"(?P<unit>{TIME_UNIT_PATTERN})\b",
    flags=re.I,
)

# A temperature unit divided by a time unit is a heating RATE, not a setpoint.
# Used on the token that follows a located unit span, so "500 °C / min" is
# skipped and the search continues to a real setpoint later in the sentence.
RAMP_TAIL_RE = re.compile(
    rf"^/\s*(?:{TIME_UNIT_PATTERN})\b",
    flags=re.I,
)

# Guard only. Part 1 writes every record as positive, so nothing in the current
# pipeline carries these labels; the check keeps a hand-edited or older record
# set from silently being trained on as if it were a positive example.
NEGATIVE_LABEL_RE = re.compile(
    r"\b(negative|near[-_\s]?miss|reject|non[-_\s]?event|false)\b",
    flags=re.I,
)

In [ ]:
# Cell 5: Canonical field keys and the aliases accepted for them.
#
# These are structured-field names, not raw-text operation words. Two naming
# systems meet here: the annotation names written by Part 1
# (operation_verb_text_span, temperature_value, time_unit) and the
# ChemDataExtractor field names a Snowball entity has to carry
# (specifier, raw_value, raw_units).
#
# ORDER MATTERS in every tuple below: lookups take the first alias that is
# present and stop, so the most specific name is listed first. Each parameter's
# tuples are wired into its ParamSpec in Cell 13.

# Where the annotated operation trigger may sit. In the current pickle it is
# event["specifier"]["text"], written by Part 1 Cell 1.
SPECIFIER_ALIASES = (
    "operation_verb",
    "operation_verb_text_span",
    "specifier",
    # "event" is deliberately not an alias: that key holds the whole event
    # clause, not the trigger word, so using it would tag the entire span as
    # the specifier.
)

# Keys looked up INSIDE a block once one of the aliases below has matched.
# "qualifier" is deliberately absent everywhere: a qualifier ("room
# temperature", "overnight") is never turned into an entity.
TEXT_KEYS = (
    "text",
    "surface",
    "normalized",
    "name",
)


# --- temperature -----------------------------------------------------------
# The nested parameter block itself, {"text", "value", "unit", "qualifier"}, is
# the live path for data written by Part 1; the flat spellings cover older,
# flatter record shapes.
TEMPERATURE_ALIASES = (
    "temperature",
    "temperature_text_span",
)

TEMPERATURE_VALUE_ALIASES = (
    "temperature_value",
    "raw_value",
)

TEMPERATURE_UNIT_ALIASES = (
    "temperature_unit",
    "raw_units",
)

TEMPERATURE_VALUE_KEYS = (
    "value",
    "text",
    "raw_value",
    "temperature_value",
    "normalized",
)

TEMPERATURE_UNIT_KEYS = (
    "unit",
    "units",
    "text",
    "raw_units",
    "temperature_unit",
    "normalized",
)

# Names looked for INSIDE a temperature block, before its "text" is parsed.
TEMPERATURE_BLOCK_VALUE_NAMES = ("raw_value", "temperature_value", "value", "temp_value")
TEMPERATURE_BLOCK_UNIT_NAMES = ("raw_units", "temperature_unit", "unit", "units", "temp_unit")


# --- time ------------------------------------------------------------------
TIME_ALIASES = (
    "duration",
    "time",
    "operation_time",
    "operation_duration",
    "calcination_time",
    "calcination_duration",
    "time_text_span",
    "duration_text_span",
)

TIME_VALUE_ALIASES = (
    "time_value",
    "duration_value",
)

TIME_UNIT_ALIASES = (
    "time_unit",
    "duration_unit",
)

TIME_VALUE_KEYS = (
    "text",
    "raw_value",
    "time_value",
    "duration_value",
    "value",
    "normalized",
)

TIME_UNIT_KEYS = (
    "text",
    "raw_units",
    "time_unit",
    "duration_unit",
    "unit",
    "units",
    "normalized",
)

TIME_BLOCK_VALUE_NAMES = ("raw_value", "time_value", "duration_value", "value")
TIME_BLOCK_UNIT_NAMES = ("raw_units", "time_unit", "duration_unit", "unit", "units")

In [ ]:
# Cell 6: Text cleanup and tokenization helpers.
#
# Everything downstream works on TOKEN indices, so the tokenizer used here has
# to be the one ChemDataExtractor uses at extraction time. Otherwise the spans
# stored in a training phrase would not line up with the tokens a learned
# pattern sees later. The one deliberate exception is the glued value+unit
# repair below, which notebook 002 has to apply as well (see Cell 3).

def as_text(value):
    """Stripped string, or None when the value is missing or blank.

    Used everywhere so a blank cell and a missing key behave identically: an
    absent candidate, never an empty string that would match anything.
    """
    if value is None:
        return None

    value = str(value).strip()
    return value if value else None


def dedupe(values):
    """Drop case-insensitive duplicates, keeping the original order.

    Order is load-bearing: candidate lists are searched first-match-wins, so the
    first spelling of a value/unit/specifier is the one that gets located.
    """
    seen = set()
    output = []

    for value in values:
        value = as_text(value)

        if not value:
            continue

        key = value.lower()

        if key not in seen:
            output.append(value)
            seen.add(key)

    return output


# A digit immediately followed by a unit, with no space: "4h", "30min", "500K".
# Single-letter units are matched case-sensitively on purpose, so "1H NMR" and
# "2D" are left alone while "4h" and "10s" are split.
GLUED_VALUE_UNIT_RE = re.compile(
    r"(?<=\d)"
    r"(?=(?:°\s*C|℃|º\s*C|oC|K|"
    r"hrs|hr|hours|hour|h|"
    r"mins|min|minutes|minute|"
    r"secs|sec|seconds|second|s|"
    r"days|day|d)"
    r"(?![A-Za-z0-9]))"
)


def normalize_for_tokenization(text):
    """Insert the space the tokenizer will not insert itself.

    The vendored ChemDataExtractor tokenizer keeps "4h" as a single token, so an
    annotated value of "4" cannot be located in a sentence that wrote "4h" and
    the record is lost as a reject. Splitting the pair apart first is what makes
    those seeds trainable; SPLIT_GLUED_VALUE_UNITS (Cell 3) turns it off.
    """
    text = as_text(text)

    if not text or not SPLIT_GLUED_VALUE_UNITS:
        return text

    return GLUED_VALUE_UNIT_RE.sub(" ", text)


def raw_tokens(text):
    """Tokenize with ChemDataExtractor and return plain token strings.

    raw_tokens keeps the surface form ("400", "°C") instead of a normalized
    token, which is what the span search has to match. It is applied to both the
    sentence and the annotated needle, so the two are always tokenized the same
    way. The loop below is the fallback for versions where Sentence exposes only
    .tokens.
    """
    sent = Sentence(normalize_for_tokenization(text) or "")

    if hasattr(sent, "raw_tokens"):
        return [str(t) for t in sent.raw_tokens]

    toks = []

    for token in sent.tokens:
        if isinstance(token, (tuple, list)):
            toks.append(str(token[0]))
        elif hasattr(token, "text"):
            toks.append(str(token.text))
        else:
            toks.append(str(token))

    return toks


def norm_token(token):
    """Fold spelling differences so two surface forms compare equal."""
    # Unicode minus / en dash / em dash all stand in for a plain hyphen in
    # reported values ("−20 °C" vs "-20 °C"). Normalising both the needle
    # and the haystack makes the match work whichever form each side uses.
    return (
        str(token)
        .replace("℃", "°C")
        .replace("º", "°")
        .replace("−", "-")
        .replace("–", "-")
        .replace("—", "-")
        .strip()
        .lower()
    )


# Annotated values are normalised to digits, but a source sentence may spell
# the number out ("for four hours", "for one day"). The digit form is always
# tried first; the spelled form is only reached when the digit is absent from
# the span.
_SPELLED_NUMBERS = {
    "0": ["zero"],
    "1": ["one"],
    "2": ["two"],
    "3": ["three"],
    "4": ["four"],
    "5": ["five"],
    "6": ["six"],
    "7": ["seven"],
    "8": ["eight"],
    "9": ["nine"],
    "10": ["ten"],
    "11": ["eleven"],
    "12": ["twelve"],
    "13": ["thirteen"],
    "14": ["fourteen"],
    "15": ["fifteen"],
    "16": ["sixteen"],
    "17": ["seventeen"],
    "18": ["eighteen"],
    "19": ["nineteen"],
    "20": ["twenty"],
}

# "400-500" and "400 to 500" are the same annotated range written two ways.
# norm_token already folds the dash characters onto "-", but it cannot turn a
# dash into the word "to", so both spellings are generated as candidates.
_RANGE_DASH_RE = re.compile(r"^([-+]?\d+(?:\.\d+)?)\s*[-–—]\s*([-+]?\d+(?:\.\d+)?)$")
_RANGE_WORD_RE = re.compile(r"^([-+]?\d+(?:\.\d+)?)\s*to\s*([-+]?\d+(?:\.\d+)?)$", re.I)


def range_variants(value):
    """Both spellings of an annotated range, or [] when the value is not one."""
    value = as_text(value)

    if not value:
        return []

    match = _RANGE_DASH_RE.match(value)

    if match:
        low, high = match.groups()
        return [f"{low} to {high}", f"{low}-{high}", f"{low} - {high}"]

    match = _RANGE_WORD_RE.match(value)

    if match:
        low, high = match.groups()
        return [f"{low}-{high}", f"{low} - {high}", f"{low} to {high}"]

    return []


def expand_value_candidates(values):
    """Append spelled-out numbers and range spellings after the plain values.

    Order is preserved: the annotated form is always tried first, and an
    expansion is only reached when the annotated form is absent from the text.
    """
    values = dedupe(values)
    expanded = list(values)

    for value in values:
        key = as_text(value)

        if key is None:
            continue

        if key.endswith(".0"):
            key = key[:-2]

        expanded.extend(_SPELLED_NUMBERS.get(key, []))
        expanded.extend(range_variants(key))

    return dedupe(expanded)


def compact_norm(tokens):
    """Glue tokens together with every space removed.

    "400 ° C" and "400°C" tokenize differently depending on the source text;
    comparing the compacted forms lets either one match the annotation.
    """
    return "".join(norm_token(t) for t in tokens).replace(" ", "")


def tokens_to_text(tokens):
    """Rebuild readable text from tokens, for messages and inspection only.

    Never used to locate a span.
    """
    text = " ".join(str(t) for t in tokens)
    text = re.sub(r"\s+([,.;:%])", r"\1", text)
    text = text.replace(" °", "°")
    return text.strip()

# Cell 7: Span-finding helpers - locate an annotated string inside a token list.
#
# Every span below is a half-open pair of TOKEN indices (start, end) into the
# sentence tokens, which is the coordinate system a Snowball Entity stores.

def find_spans(sentence_tokens, entity_text):
    """All token spans where `entity_text` occurs, loosest match last.

    Three passes, in increasing tolerance:
      1. exact token-sequence match
      2. match after unicode folding (norm_token)
      3. compacted match over a small sliding window, for text split or glued
         differently than in the annotation
    Later passes only add spans the earlier ones missed.
    """
    entity_text = as_text(entity_text)

    if not entity_text:
        return []

    needle = raw_tokens(entity_text)

    if not needle:
        return []

    spans = []
    seen = set()
    n = len(needle)

    def add(start, end):
        if (start, end) not in seen:
            spans.append((start, end))
            seen.add((start, end))

    # Pass 1: the annotated tokens appear verbatim.
    for i in range(0, len(sentence_tokens) - n + 1):
        if sentence_tokens[i:i + n] == needle:
            add(i, i + n)

    # Pass 2: same tokens after unicode folding.
    norm_sentence = [norm_token(t) for t in sentence_tokens]
    norm_needle = [norm_token(t) for t in needle]

    for i in range(0, len(norm_sentence) - n + 1):
        if norm_sentence[i:i + n] == norm_needle:
            add(i, i + n)

    # Pass 3: ignore tokenization and spacing entirely, over a window just wide
    # enough to cover the annotated text.
    target = compact_norm(needle)
    max_window = min(len(sentence_tokens), max(n + 2, 3))

    for i in range(len(sentence_tokens)):
        for width in range(1, max_window + 1):
            end = i + width

            if end > len(sentence_tokens):
                break

            if compact_norm(sentence_tokens[i:end]) == target:
                add(i, end)

    return spans


def find_span(sentence_tokens, entity_text):
    """First span of one string, or None if it cannot be located."""
    spans = find_spans(sentence_tokens, entity_text)
    return spans[0] if spans else None


def first_span(sentence_tokens, candidates):
    """First candidate that can be located, with the text that actually matched.

    Candidates are tried in list order, so dedupe() ordering decides which
    spelling wins.
    """
    for text in candidates:
        text = as_text(text)

        if not text:
            continue

        span = find_span(sentence_tokens, text)

        if span is not None:
            return text, span

    return None, None


def is_ramp_unit(sentence_tokens, unit_span):
    """True when the located unit is the numerator of a rate ("5 °C / min").

    A heating rate is not a setpoint, and ramp parameters are out of scope, so a
    unit divided by a time unit is skipped and the search continues to a real
    setpoint later in the sentence. Both tokenizations are covered: the slash
    glued to the unit token, and "/" standing as its own token.
    """
    glued = str(sentence_tokens[unit_span[1] - 1])
    slash_index = glued.find("/")

    if slash_index != -1 and RAMP_TAIL_RE.match(glued[slash_index:]):
        return True

    tail = "".join(str(t) for t in sentence_tokens[unit_span[1]:unit_span[1] + 2])

    return bool(RAMP_TAIL_RE.match(tail))


def first_span_in_slice(sentence_tokens, candidates, start, end, skip=None):
    """Search only tokens[start:end], returning the span in sentence coordinates.

    Used to require the unit to sit just after its value rather than anywhere in
    the sentence. `skip` rejects an otherwise valid occurrence - the unit search
    passes is_ramp_unit here - and the search then continues with the next
    occurrence instead of stopping at the first hit.
    """
    start = max(0, start)
    end = min(len(sentence_tokens), end)
    sliced_tokens = sentence_tokens[start:end]

    for text in candidates:
        text = as_text(text)

        if not text:
            continue

        for span in find_spans(sliced_tokens, text):
            span = (span[0] + start, span[1] + start)

            if skip is not None and skip(span):
                continue

            return text, span

    return None, None


def first_span_before_value(sentence_tokens, candidates, value_start):
    """Candidate occurrence closest before the value span.

    A sentence can name several operations ("dried at 80 °C, then calcined at
    500 °C for 4 h"). The trigger nearest to the LEFT of the value is the one
    that value belongs to. Ties break toward the longer span, so a multi-word
    trigger wins over its own first word.
    """
    matches = []

    for text in candidates:
        text = as_text(text)

        if not text:
            continue

        for span in find_spans(sentence_tokens, text):
            if span[1] <= value_start:
                distance = abs(value_start - span[1])
                length = span[1] - span[0]
                matches.append((distance, -length, text, span))

    if not matches:
        return None, None

    matches.sort()
    _, _, text, span = matches[0]

    return text, span

In [ ]:
# Cell 8: Model parse-expression lookup and nested-record readers.
#
# The readers here answer one question: for a given record, what strings did
# the annotator write for the specifier, the value and the unit? They collect
# CANDIDATES only. Nothing here decides where those strings sit in a sentence.

def field_parse_expression(model_cls, field_name):
    """Return the parse expression Snowball needs for one model field.

    An Entity stores the expression that would have produced its span, so the
    phrase can be replayed against the model later. Unit-bearing fields wrap
    their expression one level down in .field, hence the second lookup.
    """
    field = model_cls.fields[field_name]

    pe = getattr(field, "parse_expression", None)

    if pe is not None:
        return pe

    inner = getattr(field, "field", None)

    if inner is not None:
        pe = getattr(inner, "parse_expression", None)

        if pe is not None:
            return pe

    raise ValueError(
        f"Could not find parse_expression for {model_cls.__name__}.{field_name}"
    )


def source_dicts(record):
    """Dicts a value may live in, most specific first.

    Record shapes have changed over the life of this pipeline (flat event dicts,
    event + entities, paragraph-level records), so a lookup scans several
    containers in priority order: explicit entities, the event's entities, the
    event itself, then the whole record. Duplicates are removed by identity so
    the same dict is not scanned twice.
    """
    sources = []

    event = record.get("event")
    entities = record.get("entities")

    if isinstance(entities, dict):
        sources.append(entities)

    if isinstance(event, dict) and isinstance(event.get("entities"), dict):
        sources.append(event["entities"])

    if isinstance(event, dict):
        sources.append(event)

    sources.append(record)

    unique = []
    seen = set()

    for source in sources:
        if not isinstance(source, dict):
            continue

        marker = id(source)

        if marker not in seen:
            unique.append(source)
            seen.add(marker)

    return unique


def add_texts_from_value(value, keys, output):
    """Collect text out of a scalar, a list, or a nested dict.

    For a dict only the requested `keys` are followed, in the order given, so a
    block's "qualifier" is never picked up as a value.
    """
    if value is None:
        return

    if isinstance(value, dict):
        for key in keys:
            if key in value:
                add_texts_from_value(value.get(key), keys, output)

        return

    if isinstance(value, (list, tuple)):
        for item in value:
            add_texts_from_value(item, keys, output)

        return

    text = as_text(value)

    if text:
        output.append(text)


def collect_texts(record, names, keys):
    """Every text stored under any of `names`, deduped and order-preserved."""
    output = []

    for source in source_dicts(record):
        for name in names:
            if name in source:
                add_texts_from_value(source.get(name), keys, output)

    return dedupe(output)


def collect_specifier_candidates(record):
    """Annotated operation triggers plus their spacing/hyphenation variants.

    specifier_surface_candidates is defined further down in Cell 10; the name is
    resolved when this runs, so only execution order matters, not cell order.
    """
    candidates = []

    for specifier_text in collect_texts(record, SPECIFIER_ALIASES, TEXT_KEYS):
        candidates.extend(specifier_surface_candidates(specifier_text))

    return dedupe(candidates)


def collect_raw_values(record, names):
    """Raw objects (not text) stored under `names`.

    Kept unflattened so a nested parameter block can be walked by Cell 9, which
    needs the value and unit to stay paired.
    """
    output = []

    for source in source_dicts(record):
        for name in names:
            if name in source:
                output.append(source.get(name))

    return output

In [ ]:
# Cell 9: Reading a parameter block, and the two directions of unit spelling.
#
# The block readers are parameter-agnostic: they take a ParamSpec (Cell 13) and
# work the same way for temperature and for time. Only the unit helpers differ
# per parameter, and they run in opposite directions, which is why there are two
# of each:
#   clean_*_units      annotated spelling -> the canonical text the entity carries
#   *_unit_variants    canonical unit     -> every spelling a paper may print,
#                                            so the span can still be found

def parse_param_text(text, regex):
    """Every value/unit pair the spec's regex can read out of one string."""
    text = as_text(text)

    if not text:
        return [], []

    values = []
    units = []

    for match in regex.finditer(text):
        values.append(match.group("value"))
        units.append(match.group("unit"))

    return values, units


def add_param_block(block, spec, values, units):
    """Read one annotated parameter block into the value/unit lists.

    The block is whatever was stored under one of the spec's block aliases: for
    the current pickle a dict {"text", "value", "unit", "qualifier"} written by
    Part 1. The qualifier is never read: a row carrying only "room temperature"
    or "overnight" has no value/unit and must not become a training phrase.
    """
    if block is None:
        return

    if isinstance(block, (list, tuple)):
        for item in block:
            add_param_block(item, spec, values, units)

        return

    if isinstance(block, dict):
        values.extend(
            collect_texts({"event": block}, spec.block_value_names, spec.value_keys)
        )
        units.extend(
            collect_texts({"event": block}, spec.block_unit_names, spec.unit_keys)
        )

        for key in ("text", "surface", "normalized"):
            parsed_values, parsed_units = parse_param_text(block.get(key), spec.regex)
            values.extend(parsed_values)
            units.extend(parsed_units)

        return

    parsed_values, parsed_units = parse_param_text(block, spec.regex)
    values.extend(parsed_values)
    units.extend(parsed_units)


def param_candidates_from_record(record, spec):
    """All annotated values and units for one parameter, anywhere in the record."""
    values = []
    units = []

    for block in collect_raw_values(record, spec.block_aliases):
        add_param_block(block, spec, values, units)

    return dedupe(values), dedupe(units)


def clean_temperature_units(unit_text):
    """Canonicalize a temperature unit to the text the entity should carry.

    The located SPAN keeps the paper's own spelling; only the entity text is
    canonical, so "oC" in the sentence still becomes °C on the entity.
    """
    unit_text = as_text(unit_text)

    if not unit_text:
        return None

    cleaned = (
        unit_text
        .replace(" ", "")
        .replace("º", "°")
        .replace("℃", "°C")
    )

    if re.fullmatch(r"[oO][cC]", cleaned):
        return "°C"

    return cleaned


def temperature_unit_variants(unit_text):
    """Expand one annotated unit into the spellings a paper may actually print.

    The annotation says °C, but the sentence may print oC, ºC, ℃ or
    "degrees C"; the span search needs all of them to find the token.
    """
    original = as_text(unit_text)
    cleaned = clean_temperature_units(unit_text)
    candidates = [original, cleaned]
    low = (cleaned or original or "").lower().replace(" ", "")

    if low in {"°c", "c", "celsius", "degc", "degreec", "degreesc", "oc"}:
        candidates.extend([
            "°C",
            "° C",
            "℃",
            "ºC",
            "º C",
            "C",
            "oC",
            "o C",
            "degrees C",
            "degree C",
            "deg C",
            "celsius",
        ])

    if low in {"k", "kelvin"}:
        candidates.extend(["K", "kelvin", "Kelvin"])

    return dedupe(candidates)


def clean_time_units(unit_text):
    """Collapse every spelling onto the canonical unit the entity carries.

    hours/hr/hrs -> h, minutes/mins -> min, seconds/secs -> s, days -> d. Days
    stay days: the parser emits them natively and the gold standard uses them,
    so converting "4 days" to hours would put a number in the entity that is not
    in the sentence. The located span still keeps the paper's own wording.
    """
    unit_text = as_text(unit_text)

    if not unit_text:
        return None

    low = unit_text.strip().lower().rstrip(".")

    if low in {"h", "hr", "hrs", "hour", "hours"}:
        return "h"

    if low in {"min", "mins", "minute", "minutes"}:
        return "min"

    if low in {"s", "sec", "secs", "second", "seconds"}:
        return "s"

    if low in {"d", "day", "days"}:
        return "d"

    return unit_text.replace(" ", "")


def time_unit_variants(unit_text):
    """The reverse of clean_time_units: canonical unit -> spellings to search.

    An annotation of "h" still has to be located in a sentence that wrote
    "hours".
    """
    original = as_text(unit_text)
    cleaned = clean_time_units(unit_text)

    candidates = [original, cleaned]
    low = (cleaned or original or "").lower().rstrip(".")

    if low == "h":
        candidates.extend(["h", "hr", "hrs", "hour", "hours"])

    if low == "min":
        candidates.extend(["min", "mins", "minute", "minutes"])

    if low == "s":
        candidates.extend(["s", "sec", "secs", "second", "seconds"])

    if low == "d":
        candidates.extend(["d", "day", "days"])

    return dedupe(candidates)

In [ ]:
# Cell 10: Locate the annotated operation trigger in the chosen token sequence.
#
# This is the cell that binds a parameter to the right operation. When a
# sentence names several operations ("dried at 80 C, then calcined at 500 C"),
# the trigger taken is the annotated one closest BEFORE the value. No specifier
# is ever inferred from unannotated context: the only candidates are spelling
# variants of the operation_verb_text_span the annotator wrote.

def specifier_surface_candidates(text):
    """Spacing/hyphenation variants of one annotated trigger.

    Same rule as Part 2's operation_surface_variants: no new operation word is
    introduced, only a different way of writing the annotated one.
    """
    text = as_text(text)

    if not text:
        return []

    candidates = [text]

    if "-" in text:
        candidates.append(text.replace("-", " "))

    if " " in text:
        candidates.append(text.replace(" ", "-"))

    return dedupe(candidates)


def locate_annotated_specifier(sentence_tokens, specifier_candidates, raw_value_span=None):
    """Locate the annotated trigger, preferring the occurrence before the value.

    Falls back to the first occurrence anywhere in the sentence when no
    annotated trigger appears to the left of the value (a value-first phrasing
    such as "500 °C calcination").
    """
    specifier_candidates = dedupe(specifier_candidates)

    if raw_value_span is not None:
        specifier_text, specifier_span = first_span_before_value(
            sentence_tokens,
            specifier_candidates,
            raw_value_span[0],
        )

        if specifier_span is not None:
            return specifier_text, specifier_span

    return first_span(sentence_tokens, specifier_candidates)

In [ ]:
# Cell 11: Choosing the token sequence a phrase is built from, and the negative filter.
#
# For data written by Part 1 the answer is always the annotated event span, so
# resolve_context_text returns on its first branch. The paragraph-scoring path
# below it only runs for older, paragraph-level records that carry no span.
#
# Both parameters use this one function. The temperature and time runs used to
# have their own copies that disagreed about priority - one preferred
# event_text_span, the other an older "sentence" key - which meant the two
# models could be trained on different token sequences for the same event.

def split_sentences(text):
    """Split a paragraph into sentences.

    ChemDataExtractor's splitter first, so sentence boundaries match the rest of
    the pipeline; a punctuation split is the fallback when that import or parse
    fails.
    """
    text = as_text(text)

    if not text:
        return []

    try:
        from chemdataextractor.doc import Paragraph

        paragraph = Paragraph(text)
        sentences = []

        for sentence in paragraph.sentences:
            sentence_text = as_text(getattr(sentence, "text", None)) or as_text(sentence)

            if sentence_text:
                sentences.append(sentence_text)

        if sentences:
            return sentences

    except Exception:
        pass

    return [
        s.strip()
        for s in re.split(r"(?<=[.!?])\s+", text)
        if s.strip()
    ]


def best_sentence_from_paragraph(paragraph_text, candidates, regex):
    """Pick the sentence holding the most annotated candidates.

    Scoring: SENTENCE_SCORE_ANNOTATED per annotated candidate found,
    SENTENCE_SCORE_REGEX if the sentence merely looks like it holds a value of
    this parameter. Annotations therefore always outweigh the regex, which can
    only break a tie. If nothing scores, the whole paragraph is returned and the
    phrase build will usually fail loudly rather than train on a guessed
    sentence.
    """
    sentences = split_sentences(paragraph_text)

    if not sentences:
        return as_text(paragraph_text)

    scored = []

    for sentence in sentences:
        score = 0
        sentence_lower = sentence.lower()

        # Prefer sentences containing annotated specifier/value/unit candidates.
        for candidate in candidates:
            candidate = as_text(candidate)

            if candidate and candidate.lower() in sentence_lower:
                score += SENTENCE_SCORE_ANNOTATED

        # The parameter regex is only a weak fallback signal.
        # It should not create new labels by itself.
        if regex.search(sentence):
            score += SENTENCE_SCORE_REGEX

        scored.append((score, sentence))

    scored.sort(key=lambda x: x[0], reverse=True)

    if scored and scored[0][0] > 0:
        return scored[0][1]

    return as_text(paragraph_text)


def resolve_context_text(record, candidates, spec):
    """Return the text a phrase should be built from.

    Priority: the annotated event span, then older record shapes, then a
    sentence scored out of the paragraph.
    """
    candidates = candidates or []

    event = record.get("event")
    if not isinstance(event, dict):
        event = {}

    # Primary source for the current framework:
    # use the annotated event span from the Excel row.
    text = as_text(event.get("event_text_span"))
    if text:
        return text

    # Legacy fallback: source_sentence may exist in older records.
    source_sentence = event.get("source_sentence")

    if isinstance(source_sentence, dict):
        text = as_text(source_sentence.get("text"))
    else:
        text = as_text(source_sentence)

    if text:
        return text

    # Additional fallback for older flattened records.
    for source in source_dicts(record):
        for key in (
            "event_text_span",
            "sentence",
            "sentence_text",
            "event_sentence",
        ):
            text = as_text(source.get(key))

            if text:
                return text

        source_sentence = source.get("source_sentence")

        if isinstance(source_sentence, dict):
            text = as_text(source_sentence.get("text"))
        else:
            text = as_text(source_sentence)

        if text:
            return text

    # Last fallback only: paragraph-level text.
    paragraph_text = (
        as_text(record.get("paragraph_text"))
        or as_text(record.get("text"))
        or as_text(event.get("paragraph_text"))
    )

    if not paragraph_text:
        return None

    return best_sentence_from_paragraph(paragraph_text, candidates, spec.regex)


def is_negative_record(record):
    """True for anything labelled negative / near-miss / reject / false.

    Nothing in the current pipeline is labelled that way - Part 1 writes every
    record as positive - so this is a guard for hand-edited or older inputs.
    """
    labels = []

    for source in source_dicts(record):
        for key in ("label", "paragraph_label", "event_label", "class"):
            label = as_text(source.get(key))

            if label:
                labels.append(label)

    return any(NEGATIVE_LABEL_RE.search(label) for label in labels)

In [ ]:
# Cell 12: Flatten paragraph-level inputs into candidate seed records.
#
# One candidate record per annotated event. The full annotated event is kept as
# the canonical source of truth under record["event"], and the helpers above
# read operation_type, specifier, event_text_span, temperature and time from
# there. No material is read: the framework targets operation/temperature/time.

def flatten_seed_records(input_items):
    """Turn the paragraph-level pickle into one candidate record per event.

    Both shapes this pipeline has produced are accepted: the
    {"paragraph_files": [...]} JSON wrapper, and the plain list written by
    Part 1 Cell 3 where each item carries an "events" list.
    """
    if isinstance(input_items, dict):
        if isinstance(input_items.get("paragraph_files"), list):
            input_items = input_items["paragraph_files"]
        else:
            input_items = [input_items]

    records = []

    for item_index, item in enumerate(input_items):
        if not isinstance(item, dict):
            continue

        events = item.get("events")

        if isinstance(events, list):
            for event_index, event in enumerate(events):
                if not isinstance(event, dict):
                    event = {"text": event}

                # Paper-level fields (ID, year, paragraph_id, filename) are
                # copied onto every event so a reject can be traced back to the
                # training row it came from. "positive" is the only label Part 1
                # writes; the lookup exists so a labelled input still reaches
                # is_negative_record.
                record = {
                    "label": event.get("label") or item.get("paragraph_label") or "positive",
                    "paragraph_id": item.get("paragraph_id"),
                    "source_paragraph_id": item.get("source_paragraph_id"),
                    "ID": item.get("ID"),
                    "title": item.get("title"),
                    "year": item.get("year"),
                    "source_filename": item.get("filename"),
                    "filename": item.get("filename"),
                    "paragraph_text": item.get("paragraph_text") or item.get("text"),
                    "event_index": event_index,
                    "item_index": item_index,
                    "event": event,
                    "entities": event.get("entities", {}) if isinstance(event.get("entities"), dict) else {},
                }

                if not is_negative_record(record):
                    records.append(record)

        # Item is already event-shaped (an older, flat pickle): treat the item
        # itself as its own event.
        else:
            record = dict(item)
            record.setdefault("event", item)
            record.setdefault("source_filename", item.get("filename"))
            record.setdefault("paragraph_text", item.get("paragraph_text") or item.get("text"))
            record.setdefault("item_index", item_index)

            if not is_negative_record(record):
                records.append(record)

    return records


def load_seed_records(input_path=None):
    """Read the pickle and return (input_count, shuffled seed records).

    Clustering is order-dependent, so the shuffle happens here, once, with the
    fixed seed from Cell 3 - both models therefore see the same record order.
    """
    input_path = Path(input_path or INPUT_PATH)

    with open(input_path, "rb") as f:
        input_items = pickle.load(f)

    # The pickle is a list, but the JSON wrapper shape is accepted too, so the
    # count has to handle both.
    input_count = (
        len(input_items.get("paragraph_files", []))
        if isinstance(input_items, dict) and isinstance(input_items.get("paragraph_files"), list)
        else len(input_items)
    )

    seed_records = flatten_seed_records(input_items)

    if SHUFFLE_SEED_RECORDS:
        random.Random(SEED_RECORDS_RANDOM_SEED).shuffle(seed_records)

    return input_count, seed_records

In [ ]:
# Cell 13: The parameter spec - everything that differs between the two models.
#
# Temperature and time are the same extraction problem over different
# vocabulary, so they are trained by the same code and differ only in this
# object. Adding a third parameter later means writing one more spec, not one
# more copy of the pipeline.
#
# Framework contract: `name` is the stem of all three output files and the
# Snowball file_name, and `model.__name__.lower()` is the root tag Snowball
# writes into every entity tag. Notebook 003 loads the models back by these
# names - keep them unchanged.

@dataclass(frozen=True)
class ParamSpec:
    name: str                     # "OperationTemperature" / "OperationTime"
    model: type                   # model class written by Part 2
    regex: "re.Pattern"           # reads value/unit out of annotated text
    reject_noun: str              # wording used in this model's reject messages
    block_aliases: Tuple[str, ...]        # nested parameter block
    value_aliases: Tuple[str, ...]        # flat value spellings
    unit_aliases: Tuple[str, ...]         # flat unit spellings
    value_keys: Tuple[str, ...]           # keys read inside a block, for values
    unit_keys: Tuple[str, ...]            # keys read inside a block, for units
    block_value_names: Tuple[str, ...]
    block_unit_names: Tuple[str, ...]
    clean_units: Callable         # annotated spelling -> canonical entity text
    unit_variants: Callable       # canonical unit -> spellings to search for
    skip_ramp_units: bool = False  # skip "°C / min" when locating the unit

    @property
    def root_tag(self):
        return self.model.__name__.lower()

    @property
    def phrase_list_path(self):
        return Path(f"{self.name}_list.pkl")

    @property
    def model_path(self):
        return SAVE_DIR / f"{self.name}.pkl"

    @property
    def rejects_path(self):
        return Path(f"{self.name}_seed_rejects.json")


TEMPERATURE_SPEC = ParamSpec(
    name="OperationTemperature",
    model=OperationTemperature,
    regex=TEMP_RE,
    reject_noun="temperature",
    block_aliases=TEMPERATURE_ALIASES,
    value_aliases=TEMPERATURE_VALUE_ALIASES,
    unit_aliases=TEMPERATURE_UNIT_ALIASES,
    value_keys=TEMPERATURE_VALUE_KEYS,
    unit_keys=TEMPERATURE_UNIT_KEYS,
    block_value_names=TEMPERATURE_BLOCK_VALUE_NAMES,
    block_unit_names=TEMPERATURE_BLOCK_UNIT_NAMES,
    clean_units=clean_temperature_units,
    unit_variants=temperature_unit_variants,
    # Only temperature can be written as a rate ("5 °C/min"); a duration is
    # never a denominator in these papers.
    skip_ramp_units=True,
)

TIME_SPEC = ParamSpec(
    name="OperationTime",
    model=OperationTime,
    regex=TIME_RE,
    reject_noun="time",
    block_aliases=TIME_ALIASES,
    value_aliases=TIME_VALUE_ALIASES,
    unit_aliases=TIME_UNIT_ALIASES,
    value_keys=TIME_VALUE_KEYS,
    unit_keys=TIME_UNIT_KEYS,
    block_value_names=TIME_BLOCK_VALUE_NAMES,
    block_unit_names=TIME_BLOCK_UNIT_NAMES,
    clean_units=clean_time_units,
    unit_variants=time_unit_variants,
)

PARAM_SPECS = {
    "temperature": TEMPERATURE_SPEC,
    "time": TIME_SPEC,
}


# Reject classes. The distinction matters for reporting: the first is the
# framework working as designed (temperature and time are extracted
# independently, so an event with only a time is a result, not a loss), the
# other two are the seed builder failing.
REJECT_EXPECTED = "expected_no_annotation"   # this parameter is not annotated here
REJECT_LOCATE = "locate_failure"             # annotated, but not found in the text
REJECT_ANNOTATION_GAP = "annotation_gap"     # the operation trigger itself is missing


class SeedRejected(Exception):
    """One record could not become a phrase, with the reason and its class."""

    def __init__(self, reason, reject_class):
        super().__init__(reason)
        self.reason = reason
        self.reject_class = reject_class


def collect_param_candidates(record, spec):
    """Value, unit and block-text candidates for one parameter.

    Shared by the phrase builder and by has_param_annotation, so the question
    "is this parameter annotated at all?" is answered by exactly the same
    reader that later tries to locate it.
    """
    block_texts = collect_texts(record, spec.block_aliases, TEXT_KEYS)
    value_candidates = collect_texts(record, spec.value_aliases, spec.value_keys)
    unit_candidates = collect_texts(record, spec.unit_aliases, spec.unit_keys)

    block_values, block_units = param_candidates_from_record(record, spec)

    value_candidates = expand_value_candidates(value_candidates + block_values)
    unit_candidates = dedupe(unit_candidates + block_units)

    return block_texts, value_candidates, unit_candidates


def has_param_annotation(record, spec):
    """True when a record carries any annotation for this parameter.

    Used to tell the two reject classes apart, and cheap enough to call when
    inspecting or filtering the seed list.
    """
    block_texts, value_candidates, unit_candidates = collect_param_candidates(record, spec)

    return bool(block_texts or value_candidates or unit_candidates)

In [ ]:
# Cell 14: Build one Snowball Phrase from one candidate record.
#
# A phrase is produced only when the whole triple can be located in the SAME
# token sequence: annotated specifier, parameter value, parameter unit. Anything
# else raises SeedRejected, and Cell 16 records the reason AND its class in the
# reject log rather than dropping the record silently.
#
# One builder serves both parameters; everything parameter-specific arrives in
# the spec.

def build_phrase_from_record(record, spec, phrase_label, sb, stats=None):
    """Build one Snowball Phrase for the operation/<parameter> relation.

    Raises SeedRejected with the reason whenever a required entity cannot be
    located, so the caller can log the record instead of training on half a
    relation. `stats` collects the near-misses that are worth reporting even
    when the phrase is built: a unit found outside its window, or a ramp rate
    skipped.
    """
    stats = stats if stats is not None else Counter()

    specifier_candidates = collect_specifier_candidates(record)
    block_texts, raw_value_candidates, raw_unit_candidates = collect_param_candidates(
        record, spec
    )

    # Nothing at all for this parameter: expected under the framework, because
    # temperature and time are annotated independently.
    if not (block_texts or raw_value_candidates or raw_unit_candidates):
        raise SeedRejected(
            f"No annotated {spec.reject_noun} target found",
            REJECT_EXPECTED,
        )

    if not raw_value_candidates:
        raise SeedRejected(
            f"No annotated {spec.reject_noun} value candidates found",
            REJECT_EXPECTED,
        )

    if not raw_unit_candidates:
        raise SeedRejected(
            f"No annotated {spec.reject_noun} unit candidates found",
            REJECT_EXPECTED,
        )

    # Should never fire: Part 1 requires an operation trigger on every row.
    if not specifier_candidates:
        raise SeedRejected(
            "No annotated operation specifier candidates found",
            REJECT_ANNOTATION_GAP,
        )

    # The annotated strings double as the score used to pick a sentence when the
    # record has no event span of its own.
    context_candidates = (
        specifier_candidates
        + block_texts
        + raw_value_candidates
        + raw_unit_candidates
    )

    sentence = resolve_context_text(record, context_candidates, spec)

    if not sentence:
        raise SeedRejected("Missing sentence or paragraph text", REJECT_LOCATE)

    sentence_tokens = raw_tokens(sentence)

    # The value is located first: it is the least ambiguous of the three, and
    # both the unit and the specifier are then searched relative to its span.
    raw_value_text, raw_value_span = first_span(sentence_tokens, raw_value_candidates)

    if raw_value_span is None:
        raise SeedRejected(
            f"Could not locate {spec.reject_noun} raw_value in sentence. "
            f"Candidates: {raw_value_candidates}",
            REJECT_LOCATE,
        )

    unit_candidates = []

    for unit_text in raw_unit_candidates:
        unit_candidates.extend(spec.unit_variants(unit_text))

    unit_candidates = dedupe(unit_candidates)

    # Every unit spelling is tried against the same tokens, so a rate can be hit
    # several times; the span is counted once.
    skipped_ramp_spans = set()

    def skip_ramp(span):
        """Reject a unit that is the numerator of a rate, and count it."""
        if not spec.skip_ramp_units:
            return False

        if is_ramp_unit(sentence_tokens, span):
            if span not in skipped_ramp_spans:
                skipped_ramp_spans.add(span)
                stats["ramp_unit_skipped"] += 1

            return True

        return False

    # A unit belongs to the value in front of it. UNIT_SEARCH_WINDOW covers
    # "500 °C" and "500 ( °C )"; only if that fails is the rest of the sentence
    # searched, which can pick up a unit from a later clause - so that widening
    # is counted and reported at the end of the run.
    raw_units_original, raw_units_span = first_span_in_slice(
        sentence_tokens,
        unit_candidates,
        raw_value_span[1],
        raw_value_span[1] + UNIT_SEARCH_WINDOW,
        skip=skip_ramp,
    )

    if raw_units_span is None:
        raw_units_original, raw_units_span = first_span_in_slice(
            sentence_tokens,
            unit_candidates,
            raw_value_span[1],
            len(sentence_tokens),
            skip=skip_ramp,
        )

        if raw_units_span is not None:
            stats["unit_outside_window"] += 1

    if raw_units_span is None:
        raise SeedRejected(
            f"Could not locate {spec.reject_noun} raw_units in sentence. "
            f"Candidates: {unit_candidates}",
            REJECT_LOCATE,
        )

    # Span keeps the paper's spelling, entity text is canonicalized.
    raw_units_text = spec.clean_units(raw_units_original) or raw_units_original

    specifier_text, specifier_span = locate_annotated_specifier(
        sentence_tokens=sentence_tokens,
        specifier_candidates=specifier_candidates,
        raw_value_span=raw_value_span,
    )

    if specifier_span is None:
        raise SeedRejected(
            f"Could not locate annotated operation specifier in sentence. "
            f"Candidates: {specifier_candidates}",
            REJECT_LOCATE,
        )

    # The three entities of the relation, tagged "<root>__<field>" and carrying
    # the model's own parse expression so the phrase can be replayed later.
    # Material is intentionally NOT an entity: every relation is a uniform
    # specifier / value / unit triple.
    entities = [
        Entity(
            specifier_text,
            f"{spec.root_tag}__specifier",
            field_parse_expression(spec.model, "specifier"),
            specifier_span[0],
            specifier_span[1],
        ),
        Entity(
            raw_value_text,
            f"{spec.root_tag}__raw_value",
            field_parse_expression(spec.model, "raw_value"),
            raw_value_span[0],
            raw_value_span[1],
        ),
        Entity(
            raw_units_text,
            f"{spec.root_tag}__raw_units",
            field_parse_expression(spec.model, "raw_units"),
            raw_units_span[0],
            raw_units_span[1],
        ),
    ]

    # Snowball expects a relation's entities in token order.
    entities = sorted(
        entities,
        key=lambda e: (e.start, e.end, e.tag, e.text),
    )

    relation = Relation(
        entities,
        confidence=1.0,
    )

    # all_names is the list of field names Snowball expanded out of the model; a
    # phrase needs it to know which tags belong to its own root.
    detailed_model = getattr(sb, "detailed_model", {})
    all_names = list(detailed_model.keys()) if isinstance(detailed_model, dict) else []

    # Newer relex takes label/root_name/all_names as keywords; older versions
    # accept only the positional context lengths and have the same attributes
    # set afterwards.
    try:
        phrase = Phrase(
            sentence_tokens,
            [relation],
            sb.prefix_length,
            sb.suffix_length,
            confidence=1.0,
            label=str(phrase_label),
            training=True,
            root_name=spec.root_tag,
            all_names=all_names,
        )

    except TypeError:
        phrase = Phrase(
            sentence_tokens,
            [relation],
            sb.prefix_length,
            sb.suffix_length,
        )

        phrase.confidence = 1.0
        phrase.label = str(phrase_label)
        phrase.training = True
        phrase.cluster_label = None
        phrase.main_cluster_label = None

    return phrase

In [ ]:
# Cell 15: Save helpers - phrase-list export, model save, reject summaries, Snowball setup.
#
# Two different things are written at the end of a run: the seed phrase list
# (<name>_list.pkl), which is a plain list of training examples, and the full
# Snowball model (Snowball_model/<name>.pkl), which carries this run's clusters
# and patterns.


def export_phrase_list(sb, path):
    """Write a flat, relabelled copy of every training phrase.

    Phrases are reachable both directly and through their cluster, so they are
    deduped by identity. Labels are renumbered and cluster assignments cleared
    so the exported file is a clean seed list: it can be re-clustered, or loaded
    by notebook 003, without dragging this run's clustering along with it.
    """
    phrase_list = []
    seen = set()

    def add_phrase(phrase):
        marker = id(phrase)

        if marker in seen:
            return

        phrase_list.append(copy.copy(phrase))
        seen.add(marker)

    for phrase in getattr(sb, "phrases", []):
        add_phrase(phrase)

    for cluster in getattr(sb, "clusters", []):
        for phrase in getattr(cluster, "phrases", []):
            add_phrase(phrase)

    for i, phrase in enumerate(phrase_list):
        phrase.label = str(i)

        if hasattr(phrase, "cluster_label"):
            phrase.cluster_label = None

        if hasattr(phrase, "main_cluster_label"):
            phrase.main_cluster_label = None

        if hasattr(phrase, "cluster_assignments"):
            phrase.cluster_assignments = set()

    with open(path, "wb") as f:
        pickle.dump(phrase_list, f, protocol=pickle.HIGHEST_PROTOCOL)

    return len(phrase_list)


def save_snowball(sb):
    """Save the full model, with the readable dumps when the version supports it.

    details=True also writes the *_patterns / *_clusters / *_relations text
    files under Snowball_model/; older versions take no argument.
    """
    try:
        sb.save(details=True)
    except TypeError:
        sb.save()


def strip_candidates(reason):
    """Group reject messages by cause, dropping the candidate list they carry.

    Without the split, the same failure appears as dozens of unique reasons
    because each message carries its own candidate strings.
    """
    reason = str(reason or "Unknown rejection reason")
    reason = reason.split(". Candidates:", 1)[0]
    reason = reason.split(" Candidates:", 1)[0]

    return reason.strip()


def reject_reason_summary(rejects):
    """Reject counts per (class, reason), most frequent first."""
    counts = Counter(
        (reject.get("reject_class") or "unclassified", strip_candidates(reject.get("reason")))
        for reject in rejects
    )

    return [
        {"reject_class": reject_class, "reason": reason, "count": count}
        for (reject_class, reason), count in counts.most_common()
    ]


def make_reject_entry(record, exc, index, reason_counts):
    """One reject log entry, with the full event kept only for the first few.

    Identity (ID, year, paragraph_id, sentence) is always kept so any reject can
    be found again in the training sheet; the full annotated event is what makes
    the log large, so it is capped per reason by MAX_FULL_EVENTS_PER_REASON.
    """
    reason = str(exc)
    grouped = strip_candidates(reason)
    reason_counts[grouped] += 1

    entry = {
        "index": index,
        "reason": reason,
        "reject_class": getattr(exc, "reject_class", REJECT_LOCATE),
        "paragraph_id": record.get("paragraph_id"),
        "source_paragraph_id": record.get("source_paragraph_id"),
        "source_filename": record.get("source_filename"),
        "event_index": record.get("event_index"),
        "ID": record.get("ID"),
        "year": record.get("year"),
        "sentence": (
            record.get("sentence")
            or record.get("sentence_text")
            or (record.get("event") or {}).get("event_text_span")
        ),
    }

    if reason_counts[grouped] <= MAX_FULL_EVENTS_PER_REASON:
        entry["event"] = record.get("event")

    return entry


def print_training_summary(spec, input_count, seed_count, phrase_count, rejects,
                           exported_count, stats):
    """Print one compact training summary after Snowball clustering.

    Expected rejects and real failures are reported separately: the first is the
    framework working as designed, the second is what measures the seed builder.
    """
    expected = sum(1 for r in rejects if r.get("reject_class") == REJECT_EXPECTED)
    failures = len(rejects) - expected

    print(
        f"{spec.name}: "
        f"input_items={input_count}, "
        f"candidate_events={seed_count}, "
        f"phrases={phrase_count}, "
        f"rejects={len(rejects)} "
        f"(expected_no_annotation={expected}, failures={failures})"
    )

    if stats:
        print(
            "  near-misses: "
            + ", ".join(f"{key}={value}" for key, value in sorted(stats.items()))
        )

    print(
        f"  saved: {spec.phrase_list_path} ({exported_count} phrases), "
        f"{spec.model_path}, {spec.rejects_path}"
    )


def make_snowball(spec):
    """Build the Snowball driver for one spec from the Cell 3 settings.

    clustering_method is dropped when the installed relex does not accept it,
    which falls back to that version's own default.
    """
    kwargs = {
        "model": spec.model,
        "save_dir": str(SAVE_DIR),
        "file_name": spec.name,
        "prefix_length": PREFIX_LENGTH,
        "suffix_length": SUFFIX_LENGTH,
        "tsim_h": TSIM_H,
        "clustering_method": CLUSTER_METHOD,
    }

    try:
        return Snowball(**kwargs)
    except TypeError:
        kwargs.pop("clustering_method", None)
        return Snowball(**kwargs)

In [ ]:
# Cell 16: The training pipeline, run once per spec.

def train(spec):
    """Load seeds, build phrases, cluster them, and save every output.

    Every record that fails to produce a phrase is written to the spec's reject
    log with its reason, its reject class and enough identity (ID, year,
    paragraph_id, sentence) to be found again in the training sheet. A single
    failure is data, not an error; the run only fails outright when NO phrase
    could be built at all.
    """
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    input_count, seed_records = load_seed_records()

    sb = make_snowball(spec)

    phrases = []
    rejects = []
    stats = Counter()
    reason_counts = Counter()

    # One record at a time: a record that cannot be located is logged and the
    # run continues, so one bad annotation never costs the whole model.
    for i, record in enumerate(seed_records):
        try:
            phrase = build_phrase_from_record(
                record=record,
                spec=spec,
                phrase_label=len(phrases),
                sb=sb,
                stats=stats,
            )
            phrases.append(phrase)

        except Exception as exc:
            rejects.append(make_reject_entry(record, exc, i, reason_counts))

    with open(spec.rejects_path, "w", encoding="utf-8") as f:
        json.dump(rejects, f, indent=2, ensure_ascii=False)

    reject_summary = reject_reason_summary(rejects)

    if reject_summary:
        display_table(reject_summary)

    # Rejects are written before this check, so the log exists even when the run
    # fails.
    if not phrases:
        raise RuntimeError(
            f"No Snowball2 phrases were created for {spec.name}. "
            f"See {spec.rejects_path} for rejected records."
        )

    # Attach the phrases in one pass with fresh labels, then cluster once:
    # cluster_all is what turns the seed phrases into extraction patterns.
    sb.phrases = []
    sb.clusters = []
    sb.clustered = False

    for phrase in phrases:
        phrase.label = str(len(sb.phrases))
        sb.phrases.append(phrase)

    try:
        sb.cluster_all(method=CLUSTER_METHOD)
    except TypeError:
        sb.cluster_all()

    exported_count = export_phrase_list(sb, spec.phrase_list_path)
    save_snowball(sb)

    print_training_summary(
        spec=spec,
        input_count=input_count,
        seed_count=len(seed_records),
        phrase_count=len(phrases),
        rejects=rejects,
        exported_count=exported_count,
        stats=stats,
    )

    return sb

## Cell 17 — Train the OperationTemperature model

In [ ]:
# Cell 17: Train the temperature model.
#
# The guard is here because this code also runs as a standalone script. In a
# notebook __name__ is "__main__" as well, so running this cell starts training.

if __name__ == "__main__":
    train(TEMPERATURE_SPEC)

## Cell 18 — Train the OperationTime model from the same seed list

In [ ]:
# Cell 18: Train the time model.
#
# Same code, same seed list, same record order - only the spec changes. The two
# models are trained on independent subsets of the same annotated rows: an event
# with a temperature but no time simply produces no phrase here and is logged as
# an expected_no_annotation reject. That is the framework working as designed,
# not a failure.

if __name__ == "__main__":
    train(TIME_SPEC)

## Cell 19 — Check both temperature and time model outputs

In [ ]:
# Cell 19: Check both trained Snowball2 model outputs.
#
# Three questions: does every expected file exist, does each phrase list load
# and how many phrases does it hold, and what did the rejects say. The reject
# table is split by class on purpose - expected_no_annotation is the framework
# working as designed, while locate_failure and annotation_gap are the numbers
# that measure the seed builder. A large failure count concentrated on one
# reason points at the annotation or the span helpers, not at the Snowball
# settings.
#
# Paths come from the specs (Cell 13), so this cell cannot drift from the ones
# the training runs actually wrote. json, pickle, Counter and Path are imported
# in Cell 2.

SPECS_TO_CHECK = [TEMPERATURE_SPEC, TIME_SPEC]

display_table([
    {"file": str(path), "status": "FOUND" if path.exists() else "MISSING"}
    for spec in SPECS_TO_CHECK
    for path in (spec.phrase_list_path, spec.model_path, spec.rejects_path)
])


# Phrase counts: the number of training examples each model was actually built
# from, which is the seed count minus that model's rejects.
phrase_rows = []

for spec in SPECS_TO_CHECK:
    path = spec.phrase_list_path

    if not path.exists():
        phrase_rows.append({
            "model": spec.name,
            "file": str(path),
            "status": "MISSING",
            "phrase_count": None,
        })
        continue

    try:
        with path.open("rb") as f:
            phrase_list = pickle.load(f)

        phrase_rows.append({
            "model": spec.name,
            "file": str(path),
            "status": "LOADED",
            "phrase_count": len(phrase_list),
        })

    except Exception as exc:
        phrase_rows.append({
            "model": spec.name,
            "file": str(path),
            "status": f"LOAD_FAILED: {exc}",
            "phrase_count": None,
        })


display_table(phrase_rows)


reject_summary_rows = []
top_reject_reason_rows = []

for spec in SPECS_TO_CHECK:
    path = spec.rejects_path

    if not path.exists():
        reject_summary_rows.append({
            "model": spec.name,
            "file": str(path),
            "status": "MISSING",
            "reject_count": None,
            "expected_no_annotation": None,
            "failures": None,
        })
        continue

    try:
        with path.open("r", encoding="utf-8") as f:
            rejects = json.load(f)

        if not isinstance(rejects, list):
            raise TypeError("Reject file is not a list")

        class_counts = Counter(
            item.get("reject_class") or "unclassified"
            for item in rejects
            if isinstance(item, dict)
        )
        expected = class_counts.get(REJECT_EXPECTED, 0)

        # A log written before the reject classes existed carries none of them,
        # and its rejects must not be read as failures.
        classified = len(rejects) - class_counts.get("unclassified", 0)

        reject_summary_rows.append({
            "model": spec.name,
            "file": str(path),
            "status": "LOADED" if classified or not rejects else "LOADED (unclassified log - re-run to classify)",
            "reject_count": len(rejects),
            "expected_no_annotation": expected,
            "failures": len(rejects) - expected,
        })

        # Failures first, then the expected ones, each by frequency.
        reason_counts = Counter(
            (item.get("reject_class") or "unclassified", strip_candidates(item.get("reason")))
            for item in rejects
            if isinstance(item, dict)
        )

        ordered = sorted(
            reason_counts.items(),
            key=lambda kv: (kv[0][0] == REJECT_EXPECTED, -kv[1]),
        )

        for (reject_class, reason), count in ordered[:10]:
            top_reject_reason_rows.append({
                "model": spec.name,
                "reject_class": reject_class,
                "reject_reason": reason,
                "count": count,
            })

    except Exception as exc:
        reject_summary_rows.append({
            "model": spec.name,
            "file": str(path),
            "status": f"LOAD_FAILED: {exc}",
            "reject_count": None,
            "expected_no_annotation": None,
            "failures": None,
        })


display_table(reject_summary_rows)

if top_reject_reason_rows:
    display_table(top_reject_reason_rows)